# CineMatch — Top 500 Reviewers + Ratings + Film Metadata

**Owner:** Geoff (CineMatch)

## Final output schema
`data/user_ratings_enriched.csv` has these columns:
```
username, movie_title, film_slug, film_id, rating,
genre_1, genre_2, genre_3, genre_4, genre_5
```
Movie titles are properly cased (e.g. *The Hunger Games: Catching Fire*). Up to 5 genres per film.

## Pipeline (3 stages)
1. **§7 user ratings** — visit each user's `/films/ratings/` page; extract `(film_slug, film_id, rating)`.   No extra request for film_id — it's right there in `data-film-id` on the poster.
2. **§9 film metadata** — dedupe to unique films, fetch `/film/<slug>/` exactly once per film,   parse title + genres. JSON-LD primary, HTML fallback.
3. **§10 enrich** — join ratings + films into the final CSV.

## Time / disk
- Stage 1 (ratings): ~500 users × ~30 pages × 0.6s ≈ **2.5 hours**
- Stage 2 (films):   ~5–10k unique films × 0.6s ≈ **50–100 min**
- Stage 3:           seconds (it's just a join)
- Total cache disk:  ~1 GB

**Recommended first run: `MAX_USERS = 25`** — verify the pipeline works end-to-end (~10 min) before committing 3+ hours.

## §1. Install dependencies

In [ ]:
%pip install --quiet requests beautifulsoup4 pandas tqdm

In [6]:
import pandas as pd
df = pd.read_csv('Data Scrapping RESULTS/user_ratings.csv')
df.username.unique()

FileNotFoundError: [Errno 2] No such file or directory: 'Data Scrapping RESULTS/user_ratings.csv'

## §2. Imports & config
**For your first run: set `MAX_USERS = 25`.** After §6 confirms data is flowing, change to `500` and re-run §2 + §7.

In [ ]:
import json
import re
import time
import csv
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm

BASE_URL = "https://letterboxd.com"

# ---- Tune these -------------------------------------------------------------
MAX_USERS                  = 250
MAX_RATING_PAGES_PER_USER  = 50
DELAY_SECONDS              = 0.6
ADAPTIVE_BACKOFF_BASE      = 30

REVIEWERS_URL = "https://letterboxd.com/reviewers/popular/this/all-time/"
REVIEWER_PAGES_TO_WALK = 20  # 20 × 25 = 500 users

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

OUTPUT_DIR    = Path("data");         OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR     = Path("cache_users");  CACHE_DIR.mkdir(exist_ok=True)
DEBUG_DIR     = Path("debug");        DEBUG_DIR.mkdir(exist_ok=True)

USERS_CSV       = OUTPUT_DIR / "users.csv"
RATINGS_CSV     = OUTPUT_DIR / "user_ratings.csv"
FILMS_CSV       = OUTPUT_DIR / "films.csv"
ENRICHED_CSV    = OUTPUT_DIR / "user_ratings_enriched.csv"
USERNAMES_CSV   = OUTPUT_DIR / "usernames.csv"

session = requests.Session()
session.headers.update(HEADERS)

import os
print("=" * 70)
print(f"  Working dir: {os.getcwd()}")
print(f"  Output files (ABSOLUTE PATHS -- look here for your CSVs):")
print(f"    {RATINGS_CSV.resolve()}")
print(f"    {FILMS_CSV.resolve()}")
print(f"    {ENRICHED_CSV.resolve()}  <-- FINAL DELIVERABLE")
print("=" * 70)
print(f"\nMAX_USERS = {MAX_USERS}")


## §3. HTTP helper (cached + adaptive 429 backoff)

In [ ]:
_consecutive_429 = 0


def cache_path_for(url):
    h = hashlib.md5(url.encode("utf-8")).hexdigest()
    return CACHE_DIR / f"{h}.html"


def fetch(url, retries=3, use_cache=True):
    global _consecutive_429
    cp = cache_path_for(url)
    if use_cache and cp.exists() and cp.stat().st_size > 0:
        return BeautifulSoup(cp.read_text(encoding="utf-8"), "html.parser")
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=30)
            if r.status_code == 200:
                cp.write_text(r.text, encoding="utf-8")
                _consecutive_429 = 0
                time.sleep(DELAY_SECONDS)
                return BeautifulSoup(r.text, "html.parser")
            if r.status_code == 404:
                return None
            if r.status_code == 429:
                _consecutive_429 += 1
                wait = min(ADAPTIVE_BACKOFF_BASE * (2 ** (_consecutive_429 - 1)), 600)
                print(f"  [429] sleeping {wait}s")
                time.sleep(wait)
                continue
            print(f"  [{r.status_code}] {url}  (attempt {attempt + 1})")
        except requests.RequestException as exc:
            print(f"  [err] {url} -- {exc}")
        time.sleep(DELAY_SECONDS * (attempt + 2))
    return None


## §3.5. Connectivity test — RUN THIS FIRST
Verifies the scraper can actually reach Letterboxd. If this fails, nothing else will work.

In [ ]:
import requests as _req
print("Testing connectivity to Letterboxd...")

test_url = f"{BASE_URL}/film/the-grand-budapest-hotel/"
try:
    r = _req.get(test_url, headers=HEADERS, timeout=30)
    print(f"  HTTP status: {r.status_code}")
    print(f"  Body length: {len(r.text):,} chars")
    body_lower = r.text[:5000].lower()

    if r.status_code != 200:
        print(f"  ❌ FAILED. Got HTTP {r.status_code}.")
    elif "just a moment" in body_lower or "cf-browser-verification" in body_lower or "challenge-platform" in body_lower:
        print("  ❌ CLOUDFLARE BLOCK. Letterboxd is challenging your scraper.")
        print("     -> try from a different network, or wait an hour and retry.")
    elif "letterboxd" not in body_lower:
        print("  ❌ Body doesn\'t look like Letterboxd. Network is intercepting.")
    else:
        print("  ✓ Connectivity is fine. Proceed to §4.")
except Exception as exc:
    print(f"  ❌ EXCEPTION: {exc!r}")
    print("     Your network can\'t reach letterboxd.com at all.")


## §4. Collect top 500 reviewer usernames
Walks first 20 pages of `letterboxd.com/reviewers/popular/this/all-time/`. **If this returns 0 users, automatically falls back to a hardcoded list of known top reviewers** so the pipeline can still run.

In [ ]:
# Hardcoded list of confirmed prolific Letterboxd reviewers (verified to exist
# at time of writing). Used as fallback if web-scraping the popular reviewers
# page returns 0 results.
KNOWN_TOP_REVIEWERS = [
    "karsten",        # ~100k+ followers
    "davidehrlich",   # IndieWire critic
    "silentdawn",
    "lucy",
    "demi",
    "filmsbyhanna",
    "jay",
    "rcjohnso",       # Rian Johnson
    "lilfilm",        # Sean Baker
    "wimwenders",
    "theneedledrop",  # Anthony Fantano
    "patrickwillems",
    "sarahxwelch",
    "fcbarcelona",
    "mattzollerseitz",
    "James (Schaffrillas)",
    "demi adejuyigbe",
    "David Sims",
    "jonathan fujii"
]


RESERVED_USERNAMES = {
    "films", "lists", "members", "reviewers", "search", "about", "pro",
    "settings", "create-account", "sign-in", "sign-out", "year-in-review",
    "almanac", "journal", "contact", "terms", "privacy", "official",
    "showdown", "actor", "director", "studio", "country", "language",
    "genre", "theme", "mini-theme", "nanogenre", "decade", "year",
    "list", "tag", "story", "writer", "producer", "editor",
    "cinematography", "composer", "ajax", "api", "stats", "people",
    "crew", "cast", "fans", "likes", "reviews", "diary", "watchlist",
    "tv", "film", "rss", "feed", "popular", "new", "rated",
}


def extract_usernames_from_page(soup, verbose=False):
    seen, out = set(), []

    def add(name):
        if not name or name in seen or name.lower() in RESERVED_USERNAMES:
            return
        seen.add(name)
        out.append(name)

    strategies = {}
    a_count = len(out)
    for row in soup.select("table.person-table tbody tr"):
        a = row.select_one("h3.title-3 a[href]") or row.select_one("a[href^='/']")
        if a:
            href = a.get("href", "").strip("/")
            if href and "/" not in href:
                add(href)
    strategies["A (table.person-table)"] = len(out) - a_count

    b_count = len(out)
    for a in soup.select("a.avatar[href^='/'], a[class*='avatar'][href^='/']"):
        href = a.get("href", "").strip("/")
        if href and "/" not in href:
            add(href)
    strategies["B (a.avatar)"] = len(out) - b_count

    c_count = len(out)
    for a in soup.select("h3 a[href^='/'], h2 a[href^='/']"):
        href = a.get("href", "").strip("/")
        if href and "/" not in href:
            add(href)
    strategies["C (h3/h2 links)"] = len(out) - c_count

    d_count = len(out)
    for a in soup.select("a[href]"):
        m = re.match(r"^/([A-Za-z0-9_-]+)/?$", a.get("href", ""))
        if m:
            add(m.group(1))
    strategies["D (any /<slug>/ anchor)"] = len(out) - d_count

    if verbose:
        print("  username strategies:")
        for k, v in strategies.items():
            print(f"    {k:35s} -> {v}")
        print(f"    TOTAL: {len(out)}")
    return out


def collect_top_reviewers(target=MAX_USERS, pages=REVIEWER_PAGES_TO_WALK):
    seen, ordered = set(), []
    pbar = tqdm(range(1, pages + 1), desc="Reviewer pages", unit="page")
    for pg in pbar:
        url = REVIEWERS_URL if pg == 1 else REVIEWERS_URL.rstrip("/") + f"/page/{pg}/"
        soup = fetch(url)
        if soup is None:
            break
        page_users = extract_usernames_from_page(soup, verbose=(pg == 1))

        # On the FIRST page, if we got 0 users, save HTML for debugging
        if pg == 1 and len(page_users) == 0:
            print(f"\n⚠️  Page 1 returned 0 usernames. Saving HTML for inspection...")
            r_debug = requests.get(url, headers=HEADERS, timeout=30)
            (DEBUG_DIR / "reviewers_page1.html").write_text(r_debug.text, encoding="utf-8")
            print(f"   Saved to: {(DEBUG_DIR / 'reviewers_page1.html').resolve()}")
            print(f"   HTTP status: {r_debug.status_code}, body: {len(r_debug.text):,} chars")
            t_el = BeautifulSoup(r_debug.text, 'html.parser').select_one('title')
            print(f"   Page <title>: {t_el.get_text(strip=True) if t_el else 'N/A'}")

        new = 0
        for u in page_users:
            if u not in seen:
                seen.add(u); ordered.append(u); new += 1
        pbar.set_postfix(users=len(ordered), new=new)
        if new == 0:
            break
        if len(ordered) >= target:
            return ordered[:target]
    return ordered


top_users = collect_top_reviewers()
print(f"\nCollected {len(top_users)} top reviewers from web scrape")

# ---- FALLBACK: if scraping returned 0, use hardcoded known reviewers --------
if len(top_users) == 0:
    print(f"\n⚠️  Web scraping returned 0 users. Falling back to hardcoded list "
          f"of {len(KNOWN_TOP_REVIEWERS)} known top reviewers.")
    top_users = list(KNOWN_TOP_REVIEWERS)
    print("   Using:", top_users)

pd.DataFrame({"username": top_users, "rank": range(1, len(top_users) + 1)}).to_csv(USERNAMES_CSV, index=False)
print(f"\nSaved {len(top_users)} usernames to {USERNAMES_CSV.resolve()}")
print("First 10:", top_users[:10])


## §5. Per-user ratings parser
Extracts `(film_slug, film_id, rating)` triples from each ratings page. **film_id is grabbed straight off the poster's `data-film-id` attribute** — no extra HTTP needed.

In [ ]:
def parse_profile(soup, username):
    name = username
    name_el = soup.select_one("h1.title-1, div.profile-name-wrap h1, h1.primaryname, h1")
    if name_el:
        candidate = name_el.get_text(strip=True)
        if candidate and not candidate.lower().startswith("letterboxd"):
            name = candidate
    location = None
    for span in soup.select("div.profile-metadata span.label, .metadatum span.label, .metadatum .label"):
        t = span.get_text(strip=True)
        if t and "http" not in t.lower():
            location = t
            break
    return {"username": username, "name": name, "location": location}


_RATED_RE = re.compile(r"rated-(\d+)")


def _find_rating_items(soup):
    items = soup.select("ul.poster-list li, li.poster-container, li.griditem")
    items = [li for li in items if li.select_one("[data-film-slug], a[href^='/film/']")]
    if items:
        return items, "direct-selector"
    seen_ids, climbed = set(), []
    for el in soup.find_all(True):
        classes = el.get("class") or []
        if any(_RATED_RE.match(c) for c in classes):
            container = el.find_parent("li") or el.find_parent("article") or el.find_parent("div")
            if container is not None and id(container) not in seen_ids:
                seen_ids.add(id(container))
                climbed.append(container)
    if climbed:
        return climbed, "climb-from-rating"
    fallback = [li for li in soup.find_all("li") if li.select_one("a[href^='/film/']")]
    return fallback, "li-with-film-link"


def parse_ratings_page(soup, verbose=False):
    """Return list of (slug, film_id, rating) triples from a single ratings page."""
    items, strategy = _find_rating_items(soup)
    if verbose:
        print(f"   item strategy: {strategy} -> {len(items)} items")

    triples = []
    for li in items:
        # Slug + film_id (from the poster div's data-* attributes)
        slug = None
        film_id = None
        poster = li.select_one("[data-film-slug]") or li.select_one("[data-film-id]")
        if poster:
            slug = poster.get("data-film-slug")
            film_id = poster.get("data-film-id")
        if not slug:
            tl = li.select_one("[data-target-link]")
            if tl:
                m = re.match(r"/film/([^/]+)/?", tl.get("data-target-link", ""))
                if m:
                    slug = m.group(1)
        if not slug:
            a = li.select_one("a[href^='/film/']")
            if a:
                m = re.match(r"/film/([^/]+)/?", a.get("href", ""))
                if m:
                    slug = m.group(1)
        if not slug:
            continue

        # Rating
        rating = None
        for el in li.find_all(True):
            classes = el.get("class") or []
            for c in classes:
                m = _RATED_RE.match(c)
                if m:
                    try:
                        rating = int(m.group(1)) / 2.0
                    except ValueError:
                        pass
                    break
            if rating is not None:
                break

        if rating is not None:
            triples.append((slug, film_id, rating))
    return triples


def get_total_pages(soup):
    pagin = soup.select_one("div.paginate-pages")
    if not pagin:
        return 1
    nums = []
    for a in pagin.select("li a"):
        try:
            nums.append(int(a.get_text(strip=True)))
        except ValueError:
            continue
    return max(nums) if nums else 1


def scrape_user(username, max_pages=MAX_RATING_PAGES_PER_USER, verbose=False):
    """Returns ({profile dict}, [(slug, film_id, rating), ...])."""
    profile_soup = fetch(f"{BASE_URL}/{username}/")
    if profile_soup is None:
        return None, []
    profile = parse_profile(profile_soup, username)

    ratings_url = f"{BASE_URL}/{username}/films/ratings/"
    pg1 = fetch(ratings_url)
    if pg1 is None:
        return profile, []

    total = min(get_total_pages(pg1), max_pages)
    if verbose:
        print(f"   ratings pages (capped): {total}")

    triples = parse_ratings_page(pg1, verbose=verbose)
    for pg in range(2, total + 1):
        soup = fetch(f"{ratings_url}page/{pg}/")
        if soup is None:
            break
        triples.extend(parse_ratings_page(soup))
    return profile, triples


## §6. Smoke test — fully diagnostic
If `Ratings collected: 0`, paste this entire output back to me.

In [ ]:
FALLBACK_TEST_USERS = ["davidehrlich", "lucy", "karsten", "demi", "filmsbyhanna"]

try:
    if top_users:
        test_user = top_users[0]; test_source = "top_users[0]"
    else:
        test_user = FALLBACK_TEST_USERS[0]; test_source = "fallback (top_users empty)"
except NameError:
    test_user = FALLBACK_TEST_USERS[0]; test_source = "fallback (top_users not defined)"

print(f"Test user: {test_user}  ({test_source})")
print("=" * 70)

import requests as _req
diag_url = f"{BASE_URL}/{test_user}/films/ratings/"
print(f"\nStage 1 -- raw fetch of: {diag_url}")
r = _req.get(diag_url, headers=HEADERS, timeout=30)
print(f"  HTTP status:   {r.status_code}")
print(f"  Body length:   {len(r.text):,} chars")

body_lower = r.text[:5000].lower()
if "just a moment" in body_lower or "cf-browser-verification" in body_lower:
    print("  ⚠️  CLOUDFLARE CHALLENGE DETECTED")
elif "letterboxd" not in body_lower:
    print("  ⚠️  Body doesn't mention 'letterboxd'")
else:
    print("  ✓ Looks like a real Letterboxd page")

(DEBUG_DIR / "smoke_ratings.html").write_text(r.text, encoding="utf-8")
print(f"  Saved HTML -> {(DEBUG_DIR / 'smoke_ratings.html').resolve()}")

print("\nStage 2 -- selector counts:")
soup = BeautifulSoup(r.text, "html.parser")
for sel in [
    "ul.poster-list li", "li.poster-container", "[data-film-slug]",
    "[data-film-id]", "a[href^='/film/']", "[class*='rated-']",
]:
    print(f"  {sel:30s} -> {len(soup.select(sel))}")

print("\nStage 3 -- run parser:")
try:
    profile, triples = scrape_user(test_user, verbose=True)
    print(f"\n  Profile: {profile}")
    print(f"  Ratings collected: {len(triples)}")
except Exception as exc:
    print(f"  EXCEPTION: {exc!r}")
    import traceback; traceback.print_exc()
    profile, triples = None, []

if triples:
    print("\nStage 4 -- first 10 (slug, film_id, rating):")
    for slug, film_id, rating in triples[:10]:
        print(f"  {slug:50s}  id={film_id}  rating={rating}")
else:
    print("\nStage 4 -- 0 ratings parsed. First <li> on page:")
    first_li = soup.select_one("ul.poster-list li, [class*='rated-']")
    if first_li and first_li.name != "li":
        first_li = first_li.find_parent("li") or first_li
    if first_li:
        print(str(first_li)[:2000])
    else:
        print(r.text[:2000])

print("\n" + "=" * 70)


## §7. Full ratings scrape — resume-safe

In [ ]:
def load_existing_progress():
    done = set()
    user_rows, rating_rows = [], []
    if USERS_CSV.exists():
        existing = pd.read_csv(USERS_CSV)
        done.update(existing["username"].astype(str).tolist())
        user_rows = existing.to_dict("records")
        print(f"Resuming: {len(done)} users already in {USERS_CSV.name}")
    if RATINGS_CSV.exists():
        existing = pd.read_csv(RATINGS_CSV)
        rating_rows = existing.to_dict("records")
        print(f"Resuming: {len(rating_rows)} ratings already in {RATINGS_CSV.name}")
    return done, user_rows, rating_rows


def write_outputs(user_rows, rating_rows):
    pd.DataFrame(user_rows).to_csv(USERS_CSV, index=False, quoting=csv.QUOTE_MINIMAL)
    pd.DataFrame(rating_rows, columns=["username", "film_slug", "film_id", "rating"]).to_csv(
        RATINGS_CSV, index=False, quoting=csv.QUOTE_MINIMAL
    )


done, user_rows, rating_rows = load_existing_progress()
target = top_users[:MAX_USERS]
todo = [u for u in target if u not in done]
print(f"\nTarget: {len(target)}. Done: {len(done)}. To scrape: {len(todo)}")
print(f"Output will be saved to: {RATINGS_CSV.resolve()}\n")

if len(target) == 0:
    print("❌ ABORT: top_users is empty. §4 must have failed AND the hardcoded fallback is empty.")
    print("   Run §4 again, or manually set top_users = ['some', 'usernames']")
    raise SystemExit("top_users is empty")

pbar = tqdm(todo, desc="Scraping users", unit="user")
for i, username in enumerate(pbar, start=1):
    profile, triples = scrape_user(username)
    if profile is None:
        pbar.set_postfix(user=username, status="skip")
        continue
    profile["total_films_rated"] = len(triples)
    profile["date_scraped"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    user_rows.append(profile)
    for slug, film_id, rating in triples:
        rating_rows.append({"username": username, "film_slug": slug, "film_id": film_id, "rating": rating})

    pbar.set_postfix(user=username[:18], ratings=len(triples), total=len(rating_rows))
    if i % 10 == 0:
        write_outputs(user_rows, rating_rows)

write_outputs(user_rows, rating_rows)
print(f"\nDone. {len(user_rows)} users / {len(rating_rows)} ratings -> {RATINGS_CSV}")


## §8. Inspect ratings

In [ ]:
ratings_df = pd.read_csv(RATINGS_CSV)
print(f"Shape: {ratings_df.shape}")
print(f"Unique users:  {ratings_df['username'].nunique()}")
print(f"Unique films:  {ratings_df['film_slug'].nunique()}")
print(f"Films with film_id: {ratings_df['film_id'].notna().sum()} / {len(ratings_df)}")
print(f"\nRating distribution:")
print(ratings_df["rating"].value_counts().sort_index())
ratings_df.head(10)


## §9. Film metadata — title + up to 5 genres per film
**One fetch per unique film** across all users (not per rating). For ~5–10k unique films, expect 50–100 minutes.

**Genre extraction has 3 fallbacks** so it should never come back empty:
1. JSON-LD `genre` field (regex-extracted to avoid CDATA-wrapping bug)
2. HTML `#tab-genres a[href*='/films/genre/']`
3. Any `a[href*='/films/genre/']` anywhere on the page

**Title extraction has 4 fallbacks** for proper-cased titles like *The Hunger Games: Catching Fire*:
1. JSON-LD `name` field
2. `<meta property='og:title'>` (strips trailing year)
3. `<h1 class='primaryname'>` / `headline-1`
4. `<title>` tag (strips trailing year)


In [ ]:
def parse_film_jsonld(soup):
    """Extract Movie JSON-LD blob using regex (handles CDATA wrappers)."""
    for tag in soup.find_all("script", {"type": "application/ld+json"}):
        raw = tag.string or tag.get_text() or ""
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if not m:
            continue
        try:
            data = json.loads(m.group(0))
        except json.JSONDecodeError:
            continue
        if isinstance(data, dict) and data.get("@type") == "Movie":
            return data
    return {}


def parse_film_title(soup, jl):
    """Multi-fallback title extraction. Returns properly-cased title."""
    title = jl.get("name") if jl else None
    if title:
        return title.strip()

    og = soup.select_one("meta[property='og:title']")
    if og:
        text = (og.get("content") or "").strip()
        # Format: "Film Title (2014)" -- strip the year
        m = re.match(r"\s*(?:\u200E|\u202A)?(.+?)\s*\(\d{4}\)\s*$", text)
        if m:
            return m.group(1).strip()
        if text:
            return text

    for sel in ["h1.primaryname", "h1.headline-1", "h1.filmtitle", "h1"]:
        h = soup.select_one(sel)
        if h:
            text = h.get_text(strip=True)
            if text and not text.lower().startswith("letterboxd"):
                return text

    t = soup.select_one("title")
    if t:
        text = t.get_text(strip=True)
        m = re.match(r"\s*(.+?)\s*\(\d{4}\)", text)
        if m:
            return m.group(1).strip()

    return None


def parse_film_genres(soup, jl):
    """Multi-fallback genre extraction. Returns list (deduped, max 5)."""
    out, seen = [], set()

    def add(g):
        if not g:
            return
        g = g.strip()
        if not g or g in seen:
            return
        seen.add(g)
        out.append(g)

    # 1. JSON-LD
    if jl:
        g = jl.get("genre") or []
        if isinstance(g, str):
            g = [g]
        for x in g:
            add(x)

    # 2. HTML #tab-genres genre links
    if not out:
        for a in soup.select("#tab-genres a[href*='/films/genre/']"):
            add(a.get_text(strip=True))

    # 3. Any anchor pointing to /films/genre/
    if not out:
        for a in soup.select("a[href*='/films/genre/']"):
            add(a.get_text(strip=True))

    return out[:5]


def scrape_film_metadata(slug):
    """Returns dict with film_slug, title, genres list. None on fetch failure."""
    soup = fetch(f"{BASE_URL}/film/{slug}/")
    if soup is None:
        return None
    jl = parse_film_jsonld(soup)
    title = parse_film_title(soup, jl)
    genres = parse_film_genres(soup, jl)
    return {"film_slug": slug, "title": title, "genres": genres}


# ---- Run film scrape over all unique slugs in user_ratings.csv ----
ratings_df = pd.read_csv(RATINGS_CSV)
unique_films = ratings_df[["film_slug", "film_id"]].drop_duplicates()
print(f"Unique films to scrape: {len(unique_films)}")

# Resume support
existing_films = pd.DataFrame()
if FILMS_CSV.exists():
    existing_films = pd.read_csv(FILMS_CSV)
    done_slugs = set(existing_films["film_slug"].astype(str))
    print(f"Already have metadata for {len(done_slugs)} films -- will skip")
else:
    done_slugs = set()

todo = uniquez_films[~unique_films["film_slug"].isin(done_slugs)]
print(f"To scrape now: {len(todo)}\n")

film_rows = existing_films.to_dict("records") if len(existing_films) else []
no_genre_count = 0
no_title_count = 0


def write_films():
    cols = ["film_slug", "film_id", "title", "genre_1", "genre_2", "genre_3", "genre_4", "genre_5"]
    pd.DataFrame(film_rows, columns=cols).to_csv(FILMS_CSV, index=False, quoting=csv.QUOTE_MINIMAL)


pbar = tqdm(todo.itertuples(index=False), total=len(todo), desc="Films", unit="film")
for i, row in enumerate(pbar, start=1):
    slug = row.film_slug
    film_id = row.film_id
    meta = scrape_film_metadata(slug)
    if meta is None:
        film_rows.append({"film_slug": slug, "film_id": film_id, "title": None,
                          "genre_1": None, "genre_2": None, "genre_3": None, "genre_4": None, "genre_5": None})
        continue
    if not meta["title"]:
        no_title_count += 1
    if not meta["genres"]:
        no_genre_count += 1

    record = {"film_slug": slug, "film_id": film_id, "title": meta["title"]}
    for j in range(5):
        record[f"genre_{j+1}"] = meta["genres"][j] if j < len(meta["genres"]) else None
    film_rows.append(record)

    pbar.set_postfix(slug=slug[:25], no_genre=no_genre_count, no_title=no_title_count)
    if i % 50 == 0:
        write_films()

write_films()
print(f"\nDone. {len(film_rows)} film records -> {FILMS_CSV}")
print(f"  films missing title:  {no_title_count}")
print(f"  films missing genres: {no_genre_count}")

if no_genre_count > 0:
    print(f"\n⚠️  {no_genre_count} films have no genres. Inspect them in {FILMS_CSV}; if pattern is "
          "consistent, run a single film page in §6-style diagnostic and let me know what you find.")


## §10. Build the enriched CSV
Final output: `data/user_ratings_enriched.csv` with the schema you asked for.

In [ ]:
ratings_df = pd.read_csv(RATINGS_CSV)
films_df = pd.read_csv(FILMS_CSV)

# Merge on film_slug (film_id is duplicated in both -- keep ratings' since it has full coverage)
films_for_merge = films_df.drop(columns=["film_id"]) if "film_id" in films_df.columns else films_df
enriched = ratings_df.merge(films_for_merge, on="film_slug", how="left")

# Reorder columns to the requested schema
enriched = enriched.rename(columns={"title": "movie_title"})
desired = ["username", "movie_title", "film_slug", "film_id", "rating",
           "genre_1", "genre_2", "genre_3", "genre_4", "genre_5"]
enriched = enriched[[c for c in desired if c in enriched.columns]]

enriched.to_csv(ENRICHED_CSV, index=False, quoting=csv.QUOTE_MINIMAL)
print(f"Wrote {len(enriched):,} rows to {ENRICHED_CSV}")
print(f"\nColumn coverage (non-null counts):")
print(enriched.notna().sum())

print(f"\nSample 10 rows:")
enriched.head(10)


## §11. Sanity checks

In [ ]:
print("=== ROW COUNTS ===")
print(f"users.csv:                   {pd.read_csv(USERS_CSV).shape}")
print(f"user_ratings.csv:            {pd.read_csv(RATINGS_CSV).shape}")
print(f"films.csv:                   {pd.read_csv(FILMS_CSV).shape}")
print(f"user_ratings_enriched.csv:   {pd.read_csv(ENRICHED_CSV).shape}")

en = pd.read_csv(ENRICHED_CSV)
print(f"\n=== COVERAGE ===")
print(f"Rows with movie_title: {en['movie_title'].notna().sum()} / {len(en)}")
print(f"Rows with film_id:     {en['film_id'].notna().sum()} / {len(en)}")
print(f"Rows with >=1 genre:   {en['genre_1'].notna().sum()} / {len(en)}")

print(f"\n=== TOP GENRES (across all ratings) ===")
genres_long = pd.concat([en[f'genre_{i}'] for i in range(1, 6)]).dropna()
print(genres_long.value_counts().head(15))

print(f"\n=== TOP FILMS (most rated) ===")
print(en.groupby(["movie_title", "film_slug"]).size().sort_values(ascending=False).head(20))
